# Sprint 4 — Análisis de Slices con Métricas RAG

Evalúa el pipeline RAG por **categoría de consulta** (slice) con las siguientes métricas:
ROUGE-1, ROUGE-L, BLEU, Exact Match, Hit Rate@K, Context Recall.

Incluye **intervalos de confianza bootstrap** (α=0.05) por slice.

**Prerequisito:** ejecutar `notebooks/03_rag_experiment.ipynb` para generar `logs/rag_evaluation_results.json`.

## 1. Configuración

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)


def _find_root() -> Path:
    here = Path(os.getcwd())
    for candidate in [here, here.parent, here.parent.parent]:
        if (candidate / "config.yaml").exists():
            return candidate.resolve()
    colab_path = Path("/content/drive/MyDrive/assistant-itimecontrol")
    if colab_path.exists():
        return colab_path
    raise RuntimeError("config.yaml no encontrado — ajusta la ruta del proyecto")


ROOT     = _find_root()
LOGS_DIR = ROOT / "logs"
LOGS_DIR.mkdir(exist_ok=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860", "#DA8BC3"]

print(f"ROOT     = {ROOT}")
print(f"LOGS_DIR = {LOGS_DIR}")

## 2. Cargar resultados de evaluación RAG

In [ ]:
EVAL_PATH = LOGS_DIR / "rag_evaluation_results.json"

if not EVAL_PATH.exists():
    raise FileNotFoundError(
        f"{EVAL_PATH} no encontrado.\n"
        "Ejecuta primero: notebooks/03_rag_experiment.ipynb"
    )

with open(EVAL_PATH, encoding="utf-8") as f:
    eval_data = json.load(f)

model     = eval_data.get("model", "RAG")
provider  = eval_data.get("provider", "groq")
n_total   = eval_data["total_questions"]
summary   = eval_data["summary"]
results   = eval_data["results"]

print(f"Modelo   : {model}")
print(f"Proveedor: {provider}")
print(f"Preguntas: {n_total}")
print("\nMétricas globales:")
for k, v in summary.items():
    print(f"  {k:<18}: {v:.4f}")

## 3. Clasificar consultas en slices

In [ ]:
SLICE_KEYWORDS = {
    "marcaciones":   ["marcaci", "asistencia", "entrada", "salida", "registrar", "marcar", "fichar", "tardanza"],
    "permisos":      ["permiso", "vacaci", "ausencia", "licencia", "justifi", "falta"],
    "horas_extras":  ["hora extra", "horas extra", "sobretiempo", "descuento", "horas trab"],
    "reportes":      ["reporte", "informe", "exportar", "descargar", "generar reporte", "estadistica", "pdf", "excel"],
    "empleados":     ["empleado", "personal", "trabajador", "agregar", "nuevo", "alta", "colaborador"],
    "configuracion": ["configur", "parametro", "ajuste", "sistema", "turno", "horario", "clave", "contrase"],
}


def classify_slice(question: str) -> str:
    q = question.lower()
    best_slice, best_count = "general", 0
    for sl, keywords in SLICE_KEYWORDS.items():
        count = sum(1 for kw in keywords if kw in q)
        if count > best_count:
            best_count, best_slice = count, sl
    return best_slice


from src.evaluation.metrics import evaluate_rag_single

records = []
for r in results:
    m = evaluate_rag_single(r["prediction"], r["reference"], r["contexts"],
                             recall_k_values=[1, 3, 5, 10])
    records.append({
        "query":          r["question"],
        "slice":          classify_slice(r["question"]),
        "num_chunks":     r["num_chunks"],
        "rouge1":         m["rouge1"],
        "rouge2":         m["rouge2"],
        "rougeL":         m["rougeL"],
        "bleu":           m["bleu"],
        "exact_match":    m["exact_match"],
        "hit_rate":       m["hit_rate"],
        "context_recall": m["context_recall"],
        "mrr":            m["mrr"],
        "recall@1":       m["recall@1"],
        "recall@3":       m["recall@3"],
        "recall@5":       m["recall@5"],
        "recall@10":      m["recall@10"],
    })

df = pd.DataFrame(records)

print("Distribucion por slice:")
print(df["slice"].value_counts().to_string())
print(f"\nTotal de preguntas clasificadas: {len(df)}")
df.head()

## 4. Bootstrap CI por slice

In [ ]:
METRIC_COLS = ["rouge1", "rouge2", "rougeL", "bleu", "hit_rate", "context_recall", "mrr", "recall@5"]
METRIC_LABELS = {
    "rouge1":         "ROUGE-1",
    "rouge2":         "ROUGE-2",
    "rougeL":         "ROUGE-L",
    "bleu":           "BLEU",
    "exact_match":    "Exact Match",
    "hit_rate":       "Hit Rate@K",
    "context_recall": "Context Recall",
    "mrr":            "MRR",
    "recall@1":       "Recall@1",
    "recall@3":       "Recall@3",
    "recall@5":       "Recall@5",
    "recall@10":      "Recall@10",
}


def bootstrap_ci(vals: list, n_boot: int = 1000, alpha: float = 0.05):
    arr = np.asarray(vals, dtype=float)
    if len(arr) < 2:
        return (float(arr[0]), float(arr[0]))
    boots = [np.mean(np.random.choice(arr, len(arr), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boots, 100 * alpha / 2)
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return (round(float(lo), 4), round(float(hi), 4))


# Calcular estadísticas por slice
slice_rows = []
for sl in sorted(df["slice"].unique()):
    sub = df[df["slice"] == sl]
    row = {"slice": sl, "n": len(sub)}
    for col in METRIC_COLS:
        if col not in df.columns:
            row[f"{col}_mean"] = 0.0
            row[f"{col}_lo"]   = 0.0
            row[f"{col}_hi"]   = 0.0
            continue
        vals  = sub[col].tolist()
        mean  = round(np.mean(vals), 4)
        lo, hi = bootstrap_ci(vals)
        row[f"{col}_mean"] = mean
        row[f"{col}_lo"]   = lo
        row[f"{col}_hi"]   = hi
    # Estado basado en MRR y Recall@5 (retrieval) como métricas principales
    recall5 = row.get("recall@5_mean", row.get("hit_rate_mean", 0))
    row["estado"] = "problematico" if recall5 < 0.30 else "ok"
    slice_rows.append(row)

df_slices = pd.DataFrame(slice_rows)

# Tabla limpia de resumen
display_cols = ["slice", "n"] + [f"{c}_mean" for c in METRIC_COLS] + ["estado"]
rename_map   = {f"{c}_mean": METRIC_LABELS.get(c, c) for c in METRIC_COLS}
df_display   = df_slices[display_cols].rename(columns=rename_map)
pd.set_option("display.float_format", "{:.4f}".format)
print("Metricas promedio por slice:")
df_display

## 5. Visualizaciones

In [ ]:
slices_sorted = df_slices.sort_values("mrr_mean", ascending=False)["slice"].tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(f"Metricas de Retrieval por Slice — {model}\n(barras de error: IC 95% bootstrap)", fontsize=12)

plot_metrics = ["mrr", "recall@5", "hit_rate", "context_recall"]
titles       = ["MRR (Mean Reciprocal Rank)", "Recall@5", "Hit Rate@K", "Context Recall"]

for ax, col, title in zip(axes.flatten(), plot_metrics, titles):
    if f"{col}_mean" not in df_slices.columns:
        ax.set_visible(False)
        continue
    means  = [df_slices.loc[df_slices["slice"] == sl, f"{col}_mean"].values[0] for sl in slices_sorted]
    lo_err = [means[i] - df_slices.loc[df_slices["slice"] == slices_sorted[i], f"{col}_lo"].values[0]
              if f"{col}_lo" in df_slices.columns else 0
              for i in range(len(slices_sorted))]
    hi_err = [df_slices.loc[df_slices["slice"] == slices_sorted[i], f"{col}_hi"].values[0] - means[i]
              if f"{col}_hi" in df_slices.columns else 0
              for i in range(len(slices_sorted))]

    bar_colors = ["#C44E52" if m < 0.25 else "#55A868" for m in means]
    bars = ax.bar(slices_sorted, means, color=bar_colors, edgecolor="white",
                  yerr=[lo_err, hi_err], capsize=4, error_kw={"linewidth": 1.2})
    global_val = eval_data["summary"].get(col, 0)
    ax.axhline(global_val, color="navy", linestyle="--", alpha=0.6,
               label=f"Global: {global_val:.3f}")
    ax.set_ylim(0, 1.1)
    ax.set_title(title)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.035,
                f"{val:.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(str(LOGS_DIR / "slices_metricas_barras.png"), dpi=150, bbox_inches="tight")
plt.show()

# Curva Recall@K por slice
fig2, ax2 = plt.subplots(figsize=(10, 5))
k_vals = [1, 3, 5, 10]
colors_sl = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860", "#DA8BC3"]
for (sl, color) in zip(slices_sorted, colors_sl):
    rvals = [df_slices.loc[df_slices["slice"] == sl, f"recall@{k}_mean"].values[0]
             if f"recall@{k}_mean" in df_slices.columns else 0
             for k in k_vals]
    ax2.plot(k_vals, rvals, "o-", color=color, linewidth=2, label=sl)
ax2.set_xlabel("K")
ax2.set_ylabel("Recall@K")
ax2.set_title("Curva Recall@K por Slice")
ax2.set_xticks(k_vals)
ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(LOGS_DIR / "slices_recall_at_k.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Heatmap de todas las métricas por slice
heat_data = df_slices.set_index("slice")[[f"{c}_mean" for c in METRIC_COLS]].rename(
    columns={f"{c}_mean": METRIC_LABELS[c] for c in METRIC_COLS}
)

fig, ax = plt.subplots(figsize=(11, max(3, len(df_slices) * 0.7)))
im = ax.imshow(heat_data.values, cmap="YlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels(heat_data.columns, rotation=20, ha="right")
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels(heat_data.index, fontsize=10)

for i in range(len(heat_data.index)):
    for j in range(len(heat_data.columns)):
        val = heat_data.values[i, j]
        ax.text(j, i, f"{val:.3f}",
                ha="center", va="center", fontsize=9,
                color="white" if val > 0.6 else "black")

plt.colorbar(im, ax=ax, label="Score")
ax.set_title(f"Heatmap de métricas RAG por slice — {model}")
plt.tight_layout()
plt.savefig(str(LOGS_DIR / "slices_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráfico de intervalos de confianza para ROUGE-1 y Hit Rate
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Intervalos de confianza bootstrap (95%) por slice", fontsize=12)

for ax, col in zip(axes, ["rouge1", "hit_rate"]):
    df_plot = df_slices.sort_values(f"{col}_mean")
    y_pos   = range(len(df_plot))

    ax.barh(y_pos,
            df_plot[f"{col}_hi"] - df_plot[f"{col}_lo"],
            left=df_plot[f"{col}_lo"],
            height=0.4, color="lightsteelblue", alpha=0.7, label="IC 95%")
    ax.scatter(df_plot[f"{col}_mean"], y_pos,
               color="navy", zorder=5, s=60, label="Media")
    ax.axvline(summary[col], color="crimson", linestyle="--",
               label=f"Global: {summary[col]:.3f}")

    ax.set_xlim(0, 1)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(df_plot["slice"].tolist())
    ax.set_xlabel("Score")
    ax.set_title(METRIC_LABELS[col])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(LOGS_DIR / "slices_ic_bootstrap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Slices problemáticos y plan de mitigación

In [ ]:
MITIGATION_MAP = {
    "marcaciones":   "Ampliar chunks del manual sobre registro de asistencia; añadir sinónimos (marcar, fichar).",
    "permisos":      "Crear chunks dedicados a tipos de permiso y proceso de solicitud; mejorar el retriever con boost.",
    "horas_extras":  "Incluir ejemplos de cálculo automático y reglas de sobretiempo en el corpus.",
    "reportes":      "Agregar ejemplos con capturas del módulo de Reportes y pasos de exportación.",
    "empleados":     "Enriquecer con el flujo de alta de empleados paso a paso desde el manual.",
    "configuracion": "Dividir en sub-temas (turnos, horarios, perfiles) para reducir la ambigüedad semántica.",
    "general":       "Clasificar mejor las preguntas o ampliar el corpus con más ejemplos variados.",
}

problematic = df_slices[df_slices["estado"] == "problematico"].copy()

if len(problematic) == 0:
    print("Todos los slices tienen ROUGE-1 ≥ 0.40 — sin slices problemáticos.")
else:
    mit_rows = []
    for _, row in problematic.iterrows():
        mit_rows.append({
            "slice":       row["slice"],
            "n":           int(row["n"]),
            "ROUGE-1":     row["rouge1_mean"],
            "Hit Rate":    row["hit_rate_mean"],
            "prioridad":   "Alta" if row["rouge1_mean"] < 0.30 else "Media",
            "mitigación":  MITIGATION_MAP.get(row["slice"], "Revisar corpus y reformular las consultas."),
        })

    df_mit = pd.DataFrame(mit_rows)
    print(f"Slices problemáticos ({len(df_mit)}):")
    pd.set_option("display.max_colwidth", 90)
    display(df_mit)

## 7. Resumen ejecutivo

In [ ]:
best_slice  = df_slices.loc[df_slices["rouge1_mean"].idxmax(), "slice"]
worst_slice = df_slices.loc[df_slices["rouge1_mean"].idxmin(), "slice"]

print("=" * 60)
print("RESUMEN DE SLICES — iTimeControl RAG")
print("=" * 60)
print(f"Modelo         : {model}")
print(f"Proveedor      : {provider}")
print(f"Total preguntas: {n_total}")
print(f"Slices         : {len(df_slices)} categorías")
print()
print("Métricas globales:")
for col in METRIC_COLS:
    print(f"  {METRIC_LABELS[col]:<18}: {summary[col]:.4f}")
print()
print(f"Mejor slice    : {best_slice}  (ROUGE-1 = {df_slices.loc[df_slices['slice']==best_slice,'rouge1_mean'].values[0]:.4f})")
print(f"Peor slice     : {worst_slice} (ROUGE-1 = {df_slices.loc[df_slices['slice']==worst_slice,'rouge1_mean'].values[0]:.4f})")
print(f"Problemáticos  : {len(df_slices[df_slices['estado']=='problematico'])}")
print("=" * 60)

# Guardar tabla de slices a JSON
out_path = LOGS_DIR / "slices_metricas.json"
df_slices.to_json(str(out_path), orient="records", indent=2, force_ascii=False)
print(f"\nResultados guardados en: {out_path}")